# AI Gaming Content Editor — Phase 0 pipeline

Run the cells top to bottom (tap ▶ on each). C5 runs **S0 Preflight → S1 Stage-in → S2 Probe → S3 Master → S4 Analysis copy → S5 QC → S6 report** and saves everything to **My Drive › AIEditor › work › <match_id>**. **Watch `master.mp4`**; `analysis.mp4` and `analysis.wav` are small working copies for the pipeline, not for watching. S3 is slow (about 1.3–2× the recording's length). S5's checks are listed at the end of C5 and in `summary.txt`. If Colab disconnects, open this notebook again and run C1–C5: stages that already finished are skipped. If a cell says the runtime was reset, use **Runtime › Run all**.

**Before the first run**
1. Upload the recording to **My Drive › AIEditor › inbox**.
2. The repo is public, so no GitHub token is needed. Only if it is made private: in Colab open **Secrets** (key icon), add `GITHUB_TOKEN` (fine-grained token, read-only *Contents* on this repo) and switch on *Notebook access*.

Cells: **C1** mount Drive · **C2** download the code at a tag · **C3** install PyYAML if missing · **C4** choose a recording · **C5** run S0 → S5 with live progress · **C6** summary of every match and S3 encode on Drive · **C7** self-test (runs the unit tests; only when its box is ticked).

In [ ]:
#@title C1 · Mount Google Drive
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
#@title C2 · Download the pipeline code at a pinned tag
tag = "v0.6.0"  #@param {type:"string"}

import base64
import os
import subprocess
import sys

REPO_URL = "https://github.com/pjenil280505-max/ai-gaming-editor.git"
REPO_DIR = "/content/ai-gaming-editor"

try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None  # the repo is public; the secret is only needed if it becomes private

git = ["git"]
auth = None
if token:
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    git += ["-c", f"http.extraHeader=Authorization: Basic {auth}"]


def sh(cmd):
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        message = result.stderr.strip()
        for secret in (token, auth):
            if secret:
                message = message.replace(secret, "***")
        raise SystemExit(f"git failed: {message}")
    return result.stdout.strip()


if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    sh(git + ["clone", "--quiet", REPO_URL, REPO_DIR])
sh(git + ["-C", REPO_DIR, "fetch", "--quiet", "--force", "--tags", "origin"])
sh(["git", "-C", REPO_DIR, "checkout", "--quiet", "--force", tag])
sha = sh(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"])

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
for name in [m for m in sys.modules if m == "core" or m.startswith("core.")]:
    del sys.modules[name]  # a new tag takes effect without restarting the runtime
print(f"Code ready: {tag} ({sha}) in {REPO_DIR}" + (" (using GITHUB_TOKEN)" if token else ""))

In [ ]:
#@title C3 · Install PyYAML if missing
import subprocess
import sys

try:
    import yaml
    print(f"PyYAML {yaml.__version__} already installed")
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "PyYAML"], check=True)
    import yaml
    print(f"PyYAML {yaml.__version__} installed")

In [ ]:
#@title C4 · Choose a recording
#@markdown Leave **recording** empty and run this cell to list the inbox, then type a number or file name and run it again.
recording = ""  #@param {type:"string"}
target_fps = "config"  #@param ["config", "auto", "30", "60"]
force_rerun = False  #@param {type:"boolean"}

try:
    from core import run
except ModuleNotFoundError as exc:     # the runtime was reset: C2's code is gone
    print(f"Python can't find '{exc.name}': the Colab runtime was reset, so the code from C2 "
          f"is gone. Use Runtime → Run all (it runs C1 to C5 again).")
else:
    print(run.describe_choice(recording) if recording.strip() else run.inbox_listing())

In [ ]:
#@title C5 · Run S0 → S5 with live progress
NOTEBOOK_VERSION = "0.6.0"

try:
    import core
    from core import run
    from core.errors import StageError
    missing = None
except ModuleNotFoundError as exc:     # the runtime was reset: C2's code is gone
    missing = exc.name

try:
    chosen = recording.strip()
except NameError:
    chosen = ""
code_version = None if missing else getattr(core, "VERSION", "0.3.0 or older")
if missing:
    print(f"Python can't find '{missing}': the Colab runtime was reset, so the code from C2 "
          f"is gone. Use Runtime → Run all (it runs C1 to C5 again).")
elif code_version != NOTEBOOK_VERSION:
    print(f"This notebook is release {NOTEBOOK_VERSION} but C2 downloaded release {code_version}. "
          f"Set C2's tag to v{NOTEBOOK_VERSION}, run C2 again, then run C5.")
elif not chosen:
    print("Choose a recording in cell C4 first.")
else:
    try:
        result = run.run_pipeline(chosen, target_fps=target_fps, force=force_rerun,
                                  notebook_version=NOTEBOOK_VERSION)
    except StageError as err:
        print(err)

In [ ]:
#@title C6 · Summary of every match and S3 encode on Drive
try:
    from core import overview
except ModuleNotFoundError as exc:     # the runtime was reset: C2's code is gone
    print(f"Python can't find '{exc.name}': the Colab runtime was reset, so the code from C2 "
          f"is gone. Use Runtime → Run all (it runs C1 to C5 again).")
else:
    print(overview.text())

In [ ]:
#@title C7 · Self-test: run the unit tests on Colab's FFmpeg
#@markdown Takes about 5–10 minutes, so it runs only when the box is ticked.
run_self_test = False  #@param {type:"boolean"}

try:
    from core import selftest
except ModuleNotFoundError as exc:     # the runtime was reset: C2's code is gone
    print(f"Python can't find '{exc.name}': the Colab runtime was reset, so the code from C2 "
          f"is gone. Use Runtime → Run all (it runs C1 to C5 again).")
else:
    if run_self_test:
        selftest.run()
    else:
        print("Tick run_self_test and run this cell to run the unit tests (about 5–10 minutes).")